<a href="https://colab.research.google.com/github/Dhiraj0079/Deep-Learning/blob/main/exp5DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# -----------------------------
# Step 1: Mount Google Drive
# -----------------------------
from google.colab import drive
drive.mount('/content/drive')

# -----------------------------
# Step 2: Dataset Path
# -----------------------------
train_dir = '/content/drive/MyDrive/dataset/train'
val_dir = '/content/drive/MyDrive/dataset/validation'

# -----------------------------
# Step 3: Data Preparation
# -----------------------------
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

# -----------------------------
# Step 4: Load Pre-trained Model
# -----------------------------
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))

# -----------------------------
# Step 5: Freeze Base Layers
# -----------------------------
for layer in base_model.layers:
    layer.trainable = False

# -----------------------------
# Step 6: Add Classifier Head
# -----------------------------
x = base_model.output
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

# -----------------------------
# Step 7: Compile Model
# -----------------------------
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# -----------------------------
# Step 8: Train Frozen Model
# -----------------------------
print("\nTraining with Frozen Layers...")
history_frozen = model.fit(train_data, epochs=5, validation_data=val_data)

# -----------------------------
# Step 9: Fine-Tuning
# -----------------------------
for layer in base_model.layers[-4:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# -----------------------------
# Step 10: Train Fine-Tuned Model
# -----------------------------
print("\nTraining with Fine-Tuning...")
history_finetune = model.fit(train_data, epochs=5, validation_data=val_data)

# -----------------------------
# Step 11: Comparison
# -----------------------------
frozen_acc = history_frozen.history['val_accuracy'][-1]
finetune_acc = history_finetune.history['val_accuracy'][-1]

print("\n--- Comparison ---")
print(f"Frozen Model Accuracy: {frozen_acc:.4f}")
print(f"Fine-Tuned Model Accuracy: {finetune_acc:.4f}")


import matplotlib.pyplot as plt

# -----------------------------
# Plot Accuracy Graph
# -----------------------------
plt.figure()
plt.plot(history_frozen.history['accuracy'])
plt.plot(history_frozen.history['val_accuracy'])
plt.plot(history_finetune.history['accuracy'])
plt.plot(history_finetune.history['val_accuracy'])

plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')

plt.legend([
    'Frozen Train Accuracy',
    'Frozen Val Accuracy',
    'Fine-tuned Train Accuracy',
    'Fine-tuned Val Accuracy'
])

plt.show()

# -----------------------------
# Plot Loss Graph
# -----------------------------
plt.figure()
plt.plot(history_frozen.history['loss'])
plt.plot(history_frozen.history['val_loss'])
plt.plot(history_finetune.history['loss'])
plt.plot(history_finetune.history['val_loss'])

plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')

plt.legend([
    'Frozen Train Loss',
    'Frozen Val Loss',
    'Fine-tuned Train Loss',
    'Fine-tuned Val Loss'
])

plt.show()



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 40 images belonging to 2 classes.
Found 16 images belonging to 2 classes.

Training with Frozen Layers...
Epoch 1/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 25s 11s/step - accuracy: 0.3750 - loss: 2.0574 - val_accuracy: 1.0000 - val_loss: 0.0308
Epoch 2/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 24s 10s/step - accuracy: 0.8750 - loss: 0.3353 - val_accuracy: 0.5000 - val_loss: 0.3989
Epoch 3/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 27s 13s/step - accuracy: 0.7750 - loss: 0.6772 - val_accuracy: 1.0000 - val_loss: 0.0110
Epoch 4/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 25s 10s/step - accuracy: 0.9750 - loss: 0.0890 - val_accuracy: 1.0000 - val_loss: 0.0180
Epoch 5/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 23s 9s/step - accuracy: 1.0000 - loss: 0.0880 - val_accuracy: 1.0000 - val_loss: 0.0281

Training with Fine-Tuning...
Epoch 1/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 29s 23s/step - accuracy: 0.9750 - loss: 0.0812 - val_accuracy: 1.0000 - val